# CTI-SYN walkthrough — analysis & synthesis *(gated)*

**Task:** from *reconstructed inputs* (observations only), produce a threat assessment; score it
against the advisory's claim set on **recall**, **faithfulness** (judge-assisted), and
**calibration**. This is the novel contribution — and the hardest. SYN is **enabled** in
production; its input-reconstruction leakage gate is enforced at ingest via the hybrid masking
policy (notebook 07).

In [ ]:
import os, sys, json

def _find_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, "src", "glokta")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (a dir containing src/glokta)")

ROOT = _find_root(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

# Best-effort load of the repo .env so live HF calls have HF_TOKEN; no dotenv dependency.
_envp = os.path.join(ROOT, ".env")
if os.path.exists(_envp):
    for _line in open(_envp):
        _s = _line.strip()
        if _s and not _s.startswith("#") and "=" in _s:
            _k, _v = _s.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("TESTING", "1")  # relax settings validators if .env is absent

MODEL = "huggingface/meta-llama/Llama-3.1-8B-Instruct"
LIVE = bool(os.environ.get("HF_TOKEN"))
print("repo root :", ROOT)
print("model     :", MODEL)
print("LIVE calls:", LIVE, "(set HF_TOKEN to enable real inference)")

def run_model(prompt, canned, max_tokens=256):
    """Call the model live if HF_TOKEN is set, else return a canned example response."""
    if LIVE:
        from glokta.infrastructure.cti.inference import complete
        try:
            return complete(MODEL, prompt, max_tokens=max_tokens, timeout=60.0, max_retries=1)
        except Exception as exc:
            print("[live call failed -> canned]", type(exc).__name__, str(exc)[:80])
            return canned
    print("[offline -> canned response]")
    return canned

## 1. Dataflow — reconstruct inputs, build the label claim set
`reconstruct_inputs` keeps observation sections (Technical Details / IOC / Overview) and **drops
conclusions** (Summary / Attribution / ATT&CK mapping / Mitigations) so the model scores on
analysis, not summarisation. `build_claim_set` builds the label: deterministic actor/CVE/technique/
IOC claims, plus a pinned LLM for sectors/mitigations/hedges (skipped here with `judge_infer=None`).

In [ ]:
# A parsed CISA advisory (the shape parse_advisory produces). In production this is built from
# the CISA RSS feed + page text; here we use a clean in-memory example.
ADVISORY = {
    "id": "AA24-100A",
    "published": __import__("datetime").date(2024, 4, 9),
    "actor": "APT29",
    "cves": ["CVE-2024-12345"],
    "techniques": ["T1059", "T1566"],
    "text": ("APT29 conducted an espionage campaign attributed with high confidence to the "
             "Russian SVR, exploiting CVE-2024-12345 and using T1059 and T1566."),
    "sections": {
        "Summary": "APT29 attribution and high-confidence assessment (a CONCLUSION).",
        "Technical Details": "The actors used T1059 and T1566; beaconing was observed to 198.51.100.23.",
        "Indicators of Compromise": "198.51.100.23",
        "MITRE ATT&CK Techniques": "T1059, T1566 (a CONCLUSION/mapping)",
        "Mitigations": "Apply vendor patches and enforce MFA.",
    },
}

# Small in-memory reference indices. In production these come from the reference tables via
# build_technique_index() and build_taa_indices(); inline here to keep the notebook DB-free.
TECHNIQUE_INDEX = {"T1059": "T1059", "T1566": "T1566", "T1064": "T1059"}  # T1064 is revoked -> T1059
ALIAS_INDEX = {"apt29": "APT29", "cozy bear": "APT29", "the dukes": "APT29",
               "apt28": "APT28", "fancy bear": "APT28"}
RELATED_INDEX = {"APT29": {"APT28"}, "APT28": {"APT29"}}

In [ ]:
from glokta.infrastructure.cti.claim_extraction import reconstruct_inputs, build_claim_set

inputs = reconstruct_inputs(ADVISORY)
print("RECONSTRUCTED INPUTS (model sees only this):\n", inputs)
print("-" * 70)
claim_set = build_claim_set(ADVISORY, judge_infer=None)   # deterministic claims only
label = {"claims": [{"type": c.type, "value": c.value, "hedge_level": c.hedge_level}
                    for c in claim_set.claims]}
print("LABEL CLAIM SET:")
for c in label["claims"]:
    print("  ", c)

## 2. Tasking — the model writes an assessment from the inputs

In [ ]:
from glokta.infrastructure.cti.prompts import build_prompt

prompt = build_prompt("syn", inputs)
print(prompt)
print("-" * 70)
canned = ("Assessment: This activity is consistent with APT29. The campaign exploited "
          "CVE-2024-12345 using T1059 and T1566, with beaconing to 198.51.100.23. "
          "Targeting likely includes government sectors (hedged).")
response = run_model(prompt, canned=canned, max_tokens=400)
print("model assessment:\n", response)

## 3. Scoring — recall / faithfulness / calibration
`claim_set_from_text` extracts the model's claims; `score_syn` compares them to the label. The
faithfulness judge checks whether each model claim is grounded in the *inputs* — here we inject a
stub judge (`lambda claim, inputs: True`); in production this is an LLM judge (`make_grounding_judge`).

In [ ]:
from glokta.infrastructure.cti.evaluator import evaluate_item

ctx = {"alias_index": ALIAS_INDEX, "inputs": inputs, "judge": (lambda claim, inputs: True)}
scored = evaluate_item("syn", label, response, ctx)
print("primary score:", round(scored.score, 3))
print("breakdown    :", scored.breakdown)
print("model claims :", scored.parsed_output["claims"])

## 4. Why faithfulness matters
A *confidently wrong* CTI claim is worse than a miss, so faithfulness (precision of grounded
claims) is weighted highest. Below, the model hallucinates an extra actor + IOC the inputs don't
support; a discerning judge rejects them and faithfulness drops.

In [ ]:
hallucinated = response + " We also attribute this to APT28 and observed 10.0.0.99."

def strict_judge(claim, inputs):
    # ground a claim only if its value literally appears in the inputs
    return str(claim.value).lower() in inputs.lower()

ctx_strict = {"alias_index": ALIAS_INDEX, "inputs": inputs, "judge": strict_judge}
s2 = evaluate_item("syn", label, hallucinated, ctx_strict)
print("with strict judge -> recall:", round(s2.breakdown["recall"], 3),
      "faithfulness:", None if s2.breakdown["faithfulness"] is None else round(s2.breakdown["faithfulness"], 3),
      "calibration:", s2.breakdown["calibration"])

**Takeaway:** SYN keeps an objective backbone (recall + calibration) and reserves the LLM judge for the genuinely fuzzy faithfulness check. It runs in production with the leakage gate enforced at ingest (masking + drop-residue); `run_syn_pilot` remains a manual spot-check.